In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

import sys
sys.path.append(r"C:\Users\Student\Documents\GitHub\DS_Team_Credit_Risk_Assessment")

from src.preprocess import CORE_FEATURES, clean_features, build_preprocessor

In [3]:
APPROVE_THRESHOLD = 0.05
REVIEW_THRESHOLD = 0.12

RISK_TIERS = [
    (0.00, 0.03, "A"),
    (0.03, 0.06, "B"),
    (0.06, 0.12, "C"),
    (0.12, 0.20, "D"),
    (0.20, 1.01, "E"),
]

REASON_CODE_LABELS = {
    "LOAN_TO_INCOME": "High loan-to-income ratio",
    "DTI_PROXY": "High debt-to-income proxy",
    "AMT_CREDIT": "High requested credit amount",
    "AMT_ANNUITY": "High annuity burden",
    "AMT_REQ_CREDIT_BUREAU_MON": "Many recent credit bureau inquiries",
    "AMT_REQ_CREDIT_BUREAU_QRT": "Many recent credit bureau inquiries",
    "AMT_REQ_CREDIT_BUREAU_YEAR": "Many credit bureau inquiries in past year",
    "EXT_SOURCE_1": "Weak external credit score",
    "EXT_SOURCE_2": "Weak external credit score",
    "EXT_SOURCE_3": "Weak external credit score",
    "DAYS_EMPLOYED": "Short employment history",
    "EMPLOYMENT_YEARS": "Short employment history",
    "AMT_INCOME_TOTAL": "Lower income level",
    "AGE": "Younger applicant age",
    "OWN_CAR_AGE": "Older vehicle / weaker asset profile",
    "CNT_CHILDREN": "Higher number of dependents",
}

In [4]:
def select_core_features_for_scoring(df: pd.DataFrame, include_target: bool = False) -> pd.DataFrame:
    """
    In training data, TARGET exists. In application_test.csv it does not.
    This helper keeps the phase 2 feature set while allowing inference data to pass through.
    """
    required = [col for col in CORE_FEATURES if include_target or col != "TARGET"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for scoring: {missing}")
    return df[required].copy()


def load_training_data(filepath: str = "../data/home-credit-default-risk/application_train.csv"):
    df = pd.read_csv(filepath)
    df = select_core_features_for_scoring(df, include_target=True)
    df = clean_features(df)

    X = df.drop(columns=["TARGET"])
    y = df["TARGET"]

    return train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )


def load_scoring_data(filepath: str = "../data/home-credit-default-risk/application_test.csv"):
    df = pd.read_csv(filepath)

    applicant_ids = None
    if "SK_ID_CURR" in df.columns:
        applicant_ids = df["SK_ID_CURR"].copy()

    X = select_core_features_for_scoring(df, include_target=False)
    X = clean_features(X)

    if applicant_ids is None:
        applicant_ids = pd.Series(np.arange(len(X)), name="SK_ID_CURR")

    return applicant_ids, X

In [5]:
def build_reason_code_pipeline():
    return Pipeline([
        ("preprocess", build_preprocessor()),
        ("model", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])


def build_calibrated_scoring_model():
    base_pipeline = Pipeline([
        ("preprocess", build_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    return CalibratedClassifierCV(
        estimator=base_pipeline,
        method="sigmoid",
        cv=5
    )

In [6]:
def get_decision(pd_score: float) -> str:
    if pd_score < APPROVE_THRESHOLD:
        return "APPROVE"
    elif pd_score <= REVIEW_THRESHOLD:
        return "REVIEW"
    return "DECLINE"


def get_risk_tier(pd_score: float) -> str:
    for low, high, tier in RISK_TIERS:
        if low <= pd_score < high:
            return tier
    return "E"

In [7]:
# =========================
# CELL 6: Reason Code Functions
# =========================
def _safe_feature_name(feature_name: str) -> str:
    # OneHotEncoder uses names like cat__NAME_EDUCATION_TYPE_Higher education
    # Collapse categorical expansions back to base feature
    if "__" in feature_name:
        feature_name = feature_name.split("__", 1)[1]

    categorical_prefixes = [
        "CODE_GENDER_", "NAME_FAMILY_STATUS_", "NAME_EDUCATION_TYPE_",
        "NAME_INCOME_TYPE_", "OCCUPATION_TYPE_", "ORGANIZATION_TYPE_",
        "FLAG_OWN_CAR_", "FLAG_OWN_REALTY_", "NAME_HOUSING_TYPE_",
        "NAME_CONTRACT_TYPE_"
    ]
    for prefix in categorical_prefixes:
        if feature_name.startswith(prefix):
            return prefix.rstrip("_")
    return feature_name


def _reason_label(base_feature: str, contribution: float) -> str:
    if base_feature in REASON_CODE_LABELS:
        return REASON_CODE_LABELS[base_feature]

    generic_map = {
        "CODE_GENDER": "Applicant demographic category influenced score",
        "NAME_FAMILY_STATUS": "Family status influenced score",
        "NAME_EDUCATION_TYPE": "Education level influenced score",
        "NAME_INCOME_TYPE": "Income type influenced score",
        "OCCUPATION_TYPE": "Occupation category influenced score",
        "ORGANIZATION_TYPE": "Organization type influenced score",
        "FLAG_OWN_CAR": "Vehicle ownership profile influenced score",
        "FLAG_OWN_REALTY": "Real estate ownership profile influenced score",
        "NAME_HOUSING_TYPE": "Housing type influenced score",
        "NAME_CONTRACT_TYPE": "Contract type influenced score",
    }
    if base_feature in generic_map:
        return generic_map[base_feature]

    direction = "higher" if contribution > 0 else "lower"
    return f"{base_feature} contributed {direction} risk"


def generate_reason_codes(reason_pipeline: Pipeline, X_sample: pd.DataFrame, top_n: int = 3):
    preprocess = reason_pipeline.named_steps["preprocess"]
    model = reason_pipeline.named_steps["model"]

    X_transformed = preprocess.transform(X_sample)
    feature_names = preprocess.get_feature_names_out()
    coefficients = model.coef_[0]

    if hasattr(X_transformed, "toarray"):
        row_values = X_transformed.toarray()[0]
    else:
        row_values = np.asarray(X_transformed)[0]

    contributions = row_values * coefficients
    ranked_idx = np.argsort(np.abs(contributions))[::-1]

    reasons = []
    seen = set()

    for idx in ranked_idx:
        if row_values[idx] == 0:
            continue

        base_feature = _safe_feature_name(feature_names[idx])
        if base_feature in seen:
            continue

        label = _reason_label(base_feature, contributions[idx])
        reasons.append(label)
        seen.add(base_feature)

        if len(reasons) == top_n:
            break

    if not reasons:
        reasons = ["Limited signal available from selected features"]

    return reasons

In [8]:
#scoring
def score_applicant(scoring_model, reason_pipeline, sample: pd.DataFrame):
    pd_score = float(scoring_model.predict_proba(sample)[0, 1])

    return {
        "pd": round(pd_score, 6),
        "decision": get_decision(pd_score),
        "risk_tier": get_risk_tier(pd_score),
        "reason_codes": generate_reason_codes(reason_pipeline, sample, top_n=3)
    }


def score_dataset(scoring_model, reason_pipeline, applicant_ids, X_test):
    print("Scoring entire dataset at once...")

    pd_scores = scoring_model.predict_proba(X_test)[:, 1]
    decisions = [get_decision(pd) for pd in pd_scores]
    risk_tiers = [get_risk_tier(pd) for pd in pd_scores]

    results = []

    for i in range(len(X_test)):
        sample = X_test.iloc[[i]].copy()
        reasons = generate_reason_codes(reason_pipeline, sample, top_n=3)

        results.append({
            "Applicant_ID": applicant_ids.iloc[i] if hasattr(applicant_ids, "iloc") else applicant_ids[i],
            "PD": pd_scores[i],
            "Decision": decisions[i],
            "Risk_Tier": risk_tiers[i],
            "Reason_1": reasons[0] if len(reasons) > 0 else None,
            "Reason_2": reasons[1] if len(reasons) > 1 else None,
            "Reason_3": reasons[2] if len(reasons) > 2 else None,
        })

    return pd.DataFrame(results)

In [9]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

In [10]:
print("Loading training data...")
X_train, X_val, y_train, y_val = load_training_data()

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

Loading training data...
X_train shape: (246008, 33)
X_val shape: (61503, 33)


In [11]:
print("Training calibrated Random Forest for PD scoring...")
scoring_model = build_calibrated_scoring_model()
scoring_model.fit(X_train, y_train)

Training calibrated Random Forest for PD scoring...


,estimator,Pipeline(step...m_state=42))])
,method,'sigmoid'
,cv,5
,n_jobs,None
,ensemble,'auto'
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False


In [14]:
print("Training Logistic Regression for explainable reason codes...")
reason_pipeline = build_reason_code_pipeline()
reason_pipeline.fit(X_train, y_train)

print("reason_pipeline created successfully")

Training Logistic Regression for explainable reason codes...
reason_pipeline created successfully


C:\Users\Student\miniconda3\envs\credit-risk-env\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [16]:
print("Saving decisioning artifacts...")
joblib.dump(scoring_model, "../models/calibrated_scoring_model.joblib")
joblib.dump(reason_pipeline, "../models/reason_code_logreg.joblib")

print("Saved:")
print("- ../models/calibrated_scoring_model.joblib")
print("- ../models/reason_code_logreg.joblib")

Saving decisioning artifacts...
Saved:
- ../models/calibrated_scoring_model.joblib
- ../models/reason_code_logreg.joblib


In [17]:
print("Loading application_test.csv for scoring...")
applicant_ids, X_test = load_scoring_data("../data/home-credit-default-risk/application_test.csv")

print("X_test shape:", X_test.shape)

Loading application_test.csv for scoring...
X_test shape: (48744, 33)


In [18]:
predictions_df = score_dataset(scoring_model, reason_pipeline, applicant_ids, X_test)
predictions_df.head()

Scoring entire dataset at once...


,Applicant_ID,PD,Decision,Risk_Tier,Reason_1,Reason_2,Reason_3
0,100001,0.051020,REVIEW,B,AMT_GOODS_PRICE contributed lower risk,High requested credit amount,Younger applicant age
1,100005,0.092817,REVIEW,C,AMT_GOODS_PRICE contributed lower risk,Younger applicant age,High requested credit amount
2,100013,0.018241,APPROVE,A,AMT_GOODS_PRICE contributed lower risk,High requested credit amount,High annuity burden
3,100028,0.035977,APPROVE,B,AMT_GOODS_PRICE contributed lower risk,High requested credit amount,High loan-to-income ratio
4,100038,0.150444,DECLINE,D,AMT_GOODS_PRICE contributed lower risk,High requested credit amount,High loan-to-income ratio


In [19]:
predictions_df.to_csv("../reports/application_test_predictions.csv", index=False)

sample_report = predictions_df.head(5).copy()
sample_report.to_csv("../reports/sample_applicants_score_report.csv", index=False)

print("Saved:")
print("- ../reports/application_test_predictions.csv")
print("- ../reports/sample_applicants_score_report.csv")

Saved:
- ../reports/application_test_predictions.csv
- ../reports/sample_applicants_score_report.csv


In [20]:
print("\nSAMPLE APPLICANT REPORT")

for _, row in sample_report.iterrows():
    print("\n----------------------------------------")
    print(f"Applicant ID: {row['Applicant_ID']}")
    print(f"PD: {row['PD']}")
    print(f"Decision: {row['Decision']}")
    print(f"Risk Tier: {row['Risk_Tier']}")
    print("Reason Codes:")
    for col in ["Reason_1", "Reason_2", "Reason_3"]:
        if col in row and pd.notna(row[col]):
            print(f" - {row[col]}")


SAMPLE APPLICANT REPORT

----------------------------------------
Applicant ID: 100001
PD: 0.05102027223988061
Decision: REVIEW
Risk Tier: B
Reason Codes:
 - AMT_GOODS_PRICE contributed lower risk
 - High requested credit amount
 - Younger applicant age

----------------------------------------
Applicant ID: 100005
PD: 0.092817417608215
Decision: REVIEW
Risk Tier: C
Reason Codes:
 - AMT_GOODS_PRICE contributed lower risk
 - Younger applicant age
 - High requested credit amount

----------------------------------------
Applicant ID: 100013
PD: 0.018240612924122247
Decision: APPROVE
Risk Tier: A
Reason Codes:
 - AMT_GOODS_PRICE contributed lower risk
 - High requested credit amount
 - High annuity burden

----------------------------------------
Applicant ID: 100028
PD: 0.03597735011648471
Decision: APPROVE
Risk Tier: B
Reason Codes:
 - AMT_GOODS_PRICE contributed lower risk
 - High requested credit amount
 - High loan-to-income ratio

----------------------------------------
Applicant 

In [ ]:
#Optional single applicant test
sample = X_test.iloc[[0]].copy()
single_result = score_applicant(scoring_model, reason_pipeline, sample)
single_result